# Section A: Data acquisition, quality control and exploration

**Pair A:** Ronah Abucheli and Mwaniki Kelvin Maina

How conserved is the Ebola genome across *Ebolavirus* species, and can genome sequence alone separate Bundibugyo from Zaire and Sudan?

This notebook covers stages 1 to 3 of the project.

* **Needs:** internet access to NCBI
* **Saves:** `handoff_A.pkl` (cleaned genomes and details), `qc_metadata.csv`, `qc_genomes.fasta` and the stage 3 figures

The four section notebooks are run in order: A, then B and C (either order), then D. Run them in the same Colab session, or upload the earlier `handoff_*.pkl` files into `ebola_project/outputs/` first. Keep the settings cell the same in all four notebooks, with your email in `NCBI_EMAIL`.

## Setup

Installs the tools the notebook needs (Biopython, MAFFT, FastTree) and loads the Python libraries.

In [ ]:
import os, sys, re, io, json, time, gzip, shutil, subprocess, pickle, hashlib, zipfile, warnings, random, contextlib, itertools
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sys.setrecursionlimit(20000)

IN_COLAB = "google.colab" in sys.modules

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr[-2000:]}")
    return r

try:
    import Bio
except ImportError:
    sh("pip -q install biopython")
    import Bio
if not (shutil.which("mafft") and (shutil.which("fasttree") or shutil.which("FastTree"))):
    print("Installing MAFFT and FastTree (about 1 minute)...")
    sh("apt-get -qq update && apt-get -qq install -y mafft fasttree", check=False)
print("MAFFT:", shutil.which("mafft"), "| FastTree:", shutil.which("fasttree") or shutil.which("FastTree"))
print("CPU cores:", os.cpu_count(), "| Biopython", Bio.__version__)

The settings for the whole run. Put your own email in `NCBI_EMAIL` (NCBI asks for it when downloading). The other settings can stay as they are.

In [ ]:
NCBI_EMAIL = "your_email@example.com"  # put your own email here
NCBI_API_KEY = None
DOWNLOAD_METHOD = "datasets"
ENTREZ_MAX_PER_SPECIES = 2500

MIN_LEN, MAX_LEN = 17000, 20500
MAX_N_FRACTION = 0.01  # drop genomes with more than 1% unknown bases
ZAIRE_CAP_PER_OUTBREAK = 40  # keep at most this many Zaire genomes per outbreak
RANDOM_SEED = 42

EDA_KMER_K = 4
KMER_K = 6  # k-mer length used by the classifiers
ML_CLASSES = ["Bundibugyo", "Sudan", "Zaire"]
FRAGMENT_TEST = True
FRAGMENT_LENGTHS = [250, 500, 1000, 2500]
N_JOBS = -1

ALIGN_MAX_SEQS = 400  # align at most this many genomes
MAFFT_MODE = "fast"  # "fast" or "auto" (slower, more accurate)
TREE_GAMMA = False
REUSE_CACHE = True

WORKDIR = Path("ebola_project")
DATA_DIR, REF_DIR, FIG_DIR, OUT_DIR = [WORKDIR / d for d in ("data", "data/refs", "figures", "outputs")]
for d in (DATA_DIR, REF_DIR, FIG_DIR, OUT_DIR): d.mkdir(parents=True, exist_ok=True)

SPECIES = ["Zaire", "Sudan", "Bundibugyo", "Reston", "Tai Forest", "Bombali"]
SPECIES_QUERY = {
    "Zaire": ["Orthoebolavirus zairense", "Zaire ebolavirus"],
    "Sudan": ["Orthoebolavirus sudanense", "Sudan ebolavirus"],
    "Bundibugyo": ["Orthoebolavirus bundibugyoense", "Bundibugyo ebolavirus"],
    "Reston": ["Orthoebolavirus restonense", "Reston ebolavirus"],
    "Tai Forest": ["Orthoebolavirus taiense", "Tai Forest ebolavirus"],
    "Bombali": ["Orthoebolavirus bombaliense", "Bombali ebolavirus"],
}
REF_ACC = {"Zaire": "NC_002549.1", "Sudan": "NC_006432.1", "Bundibugyo": "NC_014373.1",
           "Reston": "NC_004161.1", "Tai Forest": "NC_014372.1", "Bombali": "NC_039345.1"}
GENE_ORDER = ["NP", "VP35", "VP40", "GP", "VP30", "VP24", "L"]
COLORS = {"Zaire": "#1f77b4", "Sudan": "#d62728", "Bundibugyo": "#2ca02c",
          "Reston": "#9467bd", "Tai Forest": "#ff7f0e", "Bombali": "#8c564b"}

TIMINGS, RESULTS = [], {}
@contextlib.contextmanager
def timed(label):
    t = time.time(); print(f"> {label} ...")
    try: yield
    finally:
        dt = time.time() - t; TIMINGS.append((label, dt)); print(f"< {label}: {dt/60:.1f} min ({dt:.0f} s)")

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED)

if "example.com" in NCBI_EMAIL:
    raise ValueError("Set NCBI_EMAIL (above) to your real email address before running the real download.")

Helper functions for reading and writing FASTA files, finding the genes in a reference genome and downloading data from NCBI.

In [ ]:
def read_fasta(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as fh:
        name, desc, buf = None, "", []
        for line in fh:
            if line.startswith(">"):
                if name is not None: yield name, desc, "".join(buf).upper()
                head = line[1:].strip().split(None, 1)
                name, desc, buf = head[0], (head[1] if len(head) > 1 else ""), []
            else: buf.append(line.strip())
        if name is not None: yield name, desc, "".join(buf).upper()

def write_fasta(path, items, width=80):
    with open(path, "w") as fh:
        for name, seq in items:
            fh.write(f">{name}\n")
            for i in range(0, len(seq), width): fh.write(seq[i:i+width] + "\n")

def base_acc(a): return a.split(".")[0]

def canon_gene(q):
    g = (q.get("gene", [""])[0]).strip(); prod = (q.get("product", [""])[0]).lower()
    if g.upper() in GENE_ORDER: return g.upper()
    if "nucleoprotein" in prod: return "NP"
    if "vp35" in prod or "cofactor" in prod: return "VP35"
    if "vp40" in prod or "matrix" in prod: return "VP40"
    if "vp30" in prod: return "VP30"
    if "vp24" in prod: return "VP24"
    if "glycoprotein" in prod and "secreted" not in prod and "small" not in prod: return "GP"
    if "polymerase" in prod or prod.strip() == "l": return "L"
    return None

def gene_table(rec):
    spans = {}
    for f in rec.features:
        if f.type != "CDS": continue
        g = canon_gene(f.qualifiers)
        if g is None: continue
        s, e = int(f.location.start) + 1, int(f.location.end)
        a, b = spans.get(g, (s, e)); spans[g] = (min(a, s), max(b, e))
    df = pd.DataFrame([(g, *spans[g]) for g in GENE_ORDER if g in spans], columns=["gene", "start", "end"])
    missing = [g for g in GENE_ORDER if g not in spans]
    if missing: print(f"  WARNING {rec.id}: genes not found in annotation: {missing}")
    return df

def fetch_refs():
    from Bio import Entrez, SeqIO
    Entrez.email = NCBI_EMAIL
    if NCBI_API_KEY: Entrez.api_key = NCBI_API_KEY
    refs, genes = {}, {}
    for sp, acc in REF_ACC.items():
        p = REF_DIR / f"{base_acc(acc)}.gb"
        if not p.exists():
            with Entrez.efetch(db="nuccore", id=base_acc(acc), rettype="gb", retmode="text") as h: p.write_text(h.read())
            time.sleep(0.4)
        rec = SeqIO.read(p, "genbank")
        refs[sp] = str(rec.seq).upper(); genes[sp] = gene_table(rec)
        REF_ACC[sp] = rec.id
    return refs, genes

def jget(d, *keys):
    for k in keys:
        if not isinstance(d, dict): return None
        d = d.get(k)
    return d

def download_datasets_cli():
    ds = DATA_DIR / "datasets"
    if not ds.exists():
        sh(f"curl -sSL -o {ds} https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets && chmod +x {ds}")
    records, rows = {}, []
    for sp, names in SPECIES_QUERY.items():
        got = False
        for nm in names:
            z = DATA_DIR / f"{sp.replace(' ', '_')}.zip"
            r = sh(f'{ds} download virus genome taxon "{nm}" --complete-only --filename {z}', check=False)
            if r.returncode != 0 or not z.exists(): continue
            out = DATA_DIR / f"unz_{sp.replace(' ', '_')}"; shutil.rmtree(out, ignore_errors=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(out)
            fna = next(out.rglob("genomic.fna"), None); rep = next(out.rglob("data_report.jsonl"), None)
            if fna is None: continue
            seqs = {n: (d, s) for n, d, s in read_fasta(fna)}
            reports = {}
            if rep is not None:
                for line in open(rep):
                    j = json.loads(line); reports[j.get("accession")] = j
            for acc, (desc, s) in seqs.items():
                j = reports.get(acc, {})
                rows.append(dict(accession=acc, species=sp, isolate=jget(j, "isolate", "name") or desc,
                                 collection_date=jget(j, "isolate", "collectionDate"),
                                 geo=jget(j, "location", "geographicLocation"), region=jget(j, "location", "geographicRegion"),
                                 desc=desc))
                records[acc] = s
            print(f"  {sp}: {len(seqs)} genomes (query '{nm}')"); got = True; break
        if not got: print(f"  WARNING: nothing downloaded for {sp}")
    return records, pd.DataFrame(rows)

def download_entrez():
    from Bio import Entrez, SeqIO
    Entrez.email = NCBI_EMAIL
    if NCBI_API_KEY: Entrez.api_key = NCBI_API_KEY
    records, rows = {}, []
    for sp, names in SPECIES_QUERY.items():
        for nm in names:
            term = f'"{nm}"[Organism] AND {MIN_LEN}:{MAX_LEN}[SLEN]'
            res = Entrez.read(Entrez.esearch(db="nuccore", term=term, retmax=0, usehistory="y"))
            n = int(res["Count"])
            if n > 0: break
        n = min(n, ENTREZ_MAX_PER_SPECIES); print(f"  {sp}: fetching {n} records")
        for start in range(0, n, 200):
            for attempt in range(4):
                try:
                    h = Entrez.efetch(db="nuccore", rettype="gb", retmode="text", retstart=start, retmax=min(200, n - start),
                                      webenv=res["WebEnv"], query_key=res["QueryKey"])
                    batch = list(SeqIO.parse(h, "genbank")); break
                except Exception as ex:
                    print("   retry", attempt, ex); time.sleep(3); batch = []
            for rec in batch:
                src = next((f for f in rec.features if f.type == "source"), None); q = src.qualifiers if src else {}
                records[rec.id] = str(rec.seq).upper()
                rows.append(dict(accession=rec.id, species=sp, isolate=(q.get("isolate") or q.get("strain") or [""])[0],
                                 collection_date=(q.get("collection_date") or [None])[0], geo=(q.get("country") or [None])[0],
                                 region=None, desc=rec.description))
            time.sleep(0.4)
    return records, pd.DataFrame(rows)

## 1. Data acquisition

Gets the genomes and their details from NCBI, plus the six reference genomes (one per species) with their gene positions.

Downloads the genomes and the reference genomes, or loads them if they were downloaded before. The details of each genome are kept in a table.

In [ ]:
with timed("Stage 1: data acquisition"):
    cache = DATA_DIR / "raw_ncbi.pkl"
    if REUSE_CACHE and cache.exists():
        records, raw_meta, REFS, GENES, _ra = pickle.load(open(cache, "rb")); REF_ACC.update(_ra); print("Loaded cached download:", cache)
    else:
        REFS, GENES = fetch_refs()
        try:
            if DOWNLOAD_METHOD != "datasets": raise RuntimeError("Entrez requested")
            records, raw_meta = download_datasets_cli()
            if len(records) < 50: raise RuntimeError(f"only {len(records)} genomes came back from NCBI Datasets")
        except Exception as ex:
            print(f"NCBI Datasets route not used ({ex}). Falling back to Entrez (slower, a few minutes more)...")
            records, raw_meta = download_entrez()
        for sp, a in REF_ACC.items():
            if a not in records and not any(base_acc(x) == base_acc(a) for x in records):
                records[a] = REFS[sp]; raw_meta.loc[len(raw_meta)] = dict(accession=a, species=sp, isolate="RefSeq", collection_date=None, geo=None, region=None, desc="RefSeq")
        raw_meta = raw_meta.drop_duplicates("accession").reset_index(drop=True)
        pickle.dump((records, raw_meta, REFS, GENES, dict(REF_ACC)), open(cache, "wb"))
    raw_meta.to_csv(DATA_DIR / "raw_metadata.csv", index=False)
    write_fasta(DATA_DIR / "reference_genomes.fasta", [(REF_ACC[s], REFS[s]) for s in SPECIES])

print(f"\n{len(raw_meta)} genomes downloaded")
display(raw_meta.species.value_counts().rename("genomes").to_frame().T)
print("Gene coordinates on the Zaire reference:"); display(GENES["Zaire"].T)
RESULTS["n_downloaded"] = int(len(raw_meta))


## 2. Quality control

Cleans the data so the analysis is fair.

Gives each genome an outbreak label from its country and year. Then it removes genomes that are too short or too long, have many unknown bases, or are exact copies of another genome. Finally it keeps only a limited number of Zaire genomes per outbreak so that Zaire does not dominate. The table at the end shows how many genomes each step removed.

In [ ]:
WEST_AFRICA = {"Guinea", "Sierra Leone", "Liberia", "Mali", "Nigeria", "Senegal"}
IMPORTED = {"USA", "United Kingdom", "Italy", "Spain", "Germany", "France", "Switzerland", "Norway", "Netherlands"}
SHORT = {"Democratic Republic of the Congo": "DRC", "Republic of the Congo": "RoC", "Cote d'Ivoire": "CIV",
         "Ivory Coast": "CIV", "United States": "USA"}

def assign_outbreak(species, country, year, region):
    if not country or pd.isna(year):
        return f"{species}:unknown"
    c = SHORT.get(country, country); reg = (region or "").lower()
    if species == "Zaire":
        if country in WEST_AFRICA or (country in IMPORTED and 2014 <= year <= 2016): return "Zaire:WestAfrica_2013-16"
        if c == "DRC" and year >= 2018 and (("kivu" in reg) or ("ituri" in reg) or year >= 2019): return "Zaire:DRC_Kivu_2018-20"
    return f"{species}:{c}_{int(year)}"

def parse_year(row):
    for v in (row.collection_date, row.isolate, row.desc):
        m = re.search(r"(19[5-9]\d|20[0-3]\d)", str(v)) if v is not None else None
        if m: return int(m.group(1))
    return np.nan

def parse_country(row):
    g = row.geo
    if g is None or (isinstance(g, float) and np.isnan(g)): return None
    return str(g).split(":")[0].strip() or None

def acgt_fraction(s):
    return sum(s.count(c) for c in "ACGT") / max(len(s), 1)

with timed("Stage 2: quality control"):
    meta = raw_meta.copy()
    meta["length"] = meta.accession.map(lambda a: len(records[a]))
    meta["acgt_frac"] = meta.accession.map(lambda a: acgt_fraction(records[a]))
    meta["year"] = meta.apply(parse_year, axis=1); meta["country"] = meta.apply(parse_country, axis=1)
    meta["is_ref"] = meta.accession.map(lambda a: base_acc(a) in {base_acc(x) for x in REF_ACC.values()})
    funnel = [("downloaded", len(meta))]

    m1 = meta[meta.is_ref | meta.length.between(MIN_LEN, MAX_LEN)]; funnel.append((f"length {MIN_LEN}-{MAX_LEN}", len(m1)))
    m2 = m1[m1.is_ref | (1 - m1.acgt_frac <= MAX_N_FRACTION)]; funnel.append((f"non-ACGT <= {MAX_N_FRACTION:.0%}", len(m2)))
    m2 = m2.copy(); m2["seq_hash"] = m2.accession.map(lambda a: hashlib.md5(records[a].encode()).hexdigest())
    m3 = m2.sort_values("is_ref", ascending=False).drop_duplicates("seq_hash", keep="first"); funnel.append(("exact duplicates removed", len(m3)))
    m3 = m3.copy(); m3["outbreak"] = [assign_outbreak(r.species, r.country, r.year, r.region) for r in m3.itertuples()]

    rng = np.random.default_rng(RANDOM_SEED); keep, cap_log = [], []
    for ob, grp in m3.groupby("outbreak"):
        if grp.species.iloc[0] == "Zaire" and len(grp) > ZAIRE_CAP_PER_OUTBREAK:
            refs_in = grp[grp.is_ref]; others = grp[~grp.is_ref]
            pick = others.sample(n=ZAIRE_CAP_PER_OUTBREAK - len(refs_in), random_state=int(rng.integers(1e9)))
            keep.append(pd.concat([refs_in, pick])); cap_log.append((ob, len(grp), ZAIRE_CAP_PER_OUTBREAK))
        else: keep.append(grp)
    qc = pd.concat(keep).reset_index(drop=True); funnel.append((f"Zaire capped at {ZAIRE_CAP_PER_OUTBREAK}/outbreak", len(qc)))
    SEQ = {a: records[a] for a in qc.accession}
    qc.drop(columns=["seq_hash"]).to_csv(OUT_DIR / "qc_metadata.csv", index=False)
    write_fasta(OUT_DIR / "qc_genomes.fasta", [(a, SEQ[a]) for a in qc.accession])

print(pd.DataFrame(funnel, columns=["QC step", "genomes remaining"]).to_string(index=False))
print("\nZaire outbreaks that were capped:"); display(pd.DataFrame(cap_log, columns=["outbreak", "before", "after"]))
tab = qc.groupby(["species", "outbreak"]).size().rename("genomes").reset_index()
print("Outbreak table (REVIEW THIS: these are the groups used for leave-one-outbreak-out):"); display(tab)
unk = (qc.outbreak.str.endswith(":unknown")).sum()
print(f"\nGenomes with unknown outbreak (missing country or year): {unk}")
RESULTS.update(n_qc=int(len(qc)), qc_by_species=qc.species.value_counts().to_dict(), funnel=funnel)

## 3. Exploratory analysis

A first look at the cleaned data.

Plots genome lengths and the number of genomes per species and outbreak. Then it counts short DNA words (k-mers) in each genome and uses PCA to check whether the species form separate groups.

In [ ]:
_LUT = np.full(256, -1, dtype=np.int64)
for _i, _c in enumerate("ACGT"): _LUT[ord(_c)] = _i

def kmer_index(seq, k):
    a = _LUT[np.frombuffer(seq.encode(), dtype=np.uint8)]
    n = len(a) - k + 1
    if n <= 0: return np.zeros(0, dtype=np.int64)
    idx = np.zeros(n, dtype=np.int64); bad = np.zeros(n, dtype=bool)
    for j in range(k):
        s = a[j:j+n]; bad |= s < 0; idx = idx * 4 + s
    idx[bad] = -1
    return idx

def kmer_freq(seq, k):
    idx = kmer_index(seq, k); idx = idx[idx >= 0]
    c = np.bincount(idx, minlength=4**k).astype(np.float32)
    return c / max(c.sum(), 1)

def kmer_matrix(seqs, k):
    return np.vstack([kmer_freq(s, k) for s in seqs])

def kmer_labels(k): return ["".join(p) for p in itertools.product("ACGT", repeat=k)]

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

with timed("Stage 3: exploratory analysis"):
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    sns.boxplot(data=qc, x="species", y="length", order=SPECIES, palette=COLORS, ax=ax[0]); ax[0].set_title("Genome length by species"); ax[0].tick_params(axis="x", rotation=30)
    cnt = qc.species.value_counts().reindex(SPECIES).fillna(0)
    ax[1].bar(cnt.index, cnt.values, color=[COLORS[s] for s in cnt.index]); ax[1].set_title("Genomes per species (after QC and Zaire cap)"); ax[1].tick_params(axis="x", rotation=30)
    for i, v in enumerate(cnt.values): ax[1].text(i, v, int(v), ha="center", va="bottom")
    ob = qc.groupby(["species", "outbreak"]).size().reset_index(name="n")
    ax[2].hist([ob[ob.species == s].n for s in SPECIES], bins=10, stacked=True, color=[COLORS[s] for s in SPECIES], label=SPECIES)
    ax[2].set_title("Outbreak sizes (genomes per outbreak)"); ax[2].set_xlabel("genomes"); ax[2].legend(fontsize=8)
    savefig("03_lengths_counts")

    X_eda = kmer_matrix([SEQ[a] for a in qc.accession], EDA_KMER_K)
    pca = PCA(n_components=4, random_state=RANDOM_SEED); P = pca.fit_transform(StandardScaler().fit_transform(X_eda))
    fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
    for s in SPECIES:
        m = (qc.species == s).values; ax[0].scatter(P[m, 0], P[m, 1], s=18, alpha=.7, label=f"{s} (n={m.sum()})", color=COLORS[s])
    ax[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})"); ax[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
    ax[0].set_title(f"PCA of {EDA_KMER_K}-mer profiles, coloured by species"); ax[0].legend(fontsize=8)
    z = (qc.species == "Zaire").values
    for obk, grp in qc[z].groupby("outbreak"):
        mm = qc.index.isin(grp.index); ax[1].scatter(P[mm, 0], P[mm, 1], s=18, alpha=.8, label=obk.split(":")[1])
    ax[1].set_title("Zaire only, coloured by outbreak"); ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2"); ax[1].legend(fontsize=6, ncol=2)
    savefig("03_kmer_pca")
print("Variance explained by PC1-4:", np.round(pca.explained_variance_ratio_, 3))
print(qc.groupby("species").length.describe()[["count", "min", "mean", "max"]].round(0))

Saves the cleaned genomes and details so that Sections B and C can use them.

In [ ]:
pickle.dump(dict(qc=qc, SEQ=SEQ, REFS=REFS, GENES=GENES, REF_ACC=dict(REF_ACC), RESULTS=RESULTS, TIMINGS=TIMINGS, funnel=funnel),
            open(OUT_DIR / "handoff_A.pkl", "wb"))
print("Saved", OUT_DIR / "handoff_A.pkl")